# 03 — Préparation dataset NER (**6 labels RoBERTa**)

## Architecture cible

| Entité | Entraînement NER | Production |
|--------|------------------|------------|
| SKILL, JOB_TITLE, COMPANY, DATE, DEGREE, INSTITUTION | **Oui** (RoBERTa-large) | NER |
| EXPERIENCE_DESC | **Non** (filtré du .spacy) | LLM Groq |
| CERTIFICATION | **Non** (filtré du .spacy) | Retirée |

## Sources
S1 Dataturks + S2 filtré + **S3 Doccano ~400 CV** (`TRAIN_MODE = "doccano_only"`)

## Sorties
- `bloc3_balanced_train.spacy` / `bloc3_balanced_dev.spacy`

## Après correction Doccano (split fixe)

**Option A — export complet ~400 CV (recommandé)** :
1. Export Doccano → `source3_annotated_merged.jsonl` (~400 CV corrigés)
2. Garde `DEV.jsonl` (60 CV dev) sur Drive
3. Supprime `TRAIN.jsonl` (20 CV partiels — plus nécessaire)

**Option B — partiel (actuel)** :
- `TRAIN.jsonl` (20) + `DEV.jsonl` (60) + fallback batch2

→ Notebook **03** puis **06** (entraînement + évaluation)


In [1]:
#@title 🔧 Configuration Google Colab + Google Drive

from google.colab import drive

import os
import sys
import json
import random
import re
import hashlib
import numpy as np

from pathlib import Path
from collections import Counter, defaultdict

drive.mount('/content/drive')

# =============================================================================
# CONFIGURATION PROJET
# =============================================================================

PROJECT = '/content/drive/MyDrive/cvextraction'
BASE = os.path.join(PROJECT, 'Datasets')

OUT = os.path.join(BASE, 'bloc1_processed')
OUT2 = os.path.join(BASE, 'bloc2_processed')
OUT3 = os.path.join(BASE, 'bloc3_processed')

for p in [BASE, OUT, OUT2, OUT3]:
    os.makedirs(p, exist_ok=True)

if not os.path.isdir(BASE):
    raise FileNotFoundError(
        f"Datasets introuvable : {BASE}\n"
        "Uploade les datasets dans :\n"
        "MyDrive/cvextraction/Datasets/"
    )

print("✅ Drive monté")
print("PROJECT :", PROJECT)
print("BASE    :", BASE)

# =============================================================================
# SEED GLOBAL
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# =============================================================================
# PARAMÈTRES DATASET
# =============================================================================

TRAIN_MODE = "doccano_only"  # doccano_only | doccano_plus | full
S2_TARGET = 80          # top CV S2 les plus diversifiés (qualité > quantité)
SYNTHETIC_TARGET = 50
S3_WEIGHT = 1           # Doccano prioritaire via SOURCE_RANK (pas de duplication liste)

# Exports Doccano corrigés (split fixe train / dev)
# Laisse "" pour auto-détection sous Datasets/ (recherche train.jsonl + dev.jsonl)
DOCCANO_TRAIN_JSONL = os.path.join(OUT, "TRAIN.jsonl")
DOCCANO_DEV_JSONL = os.path.join(OUT, "DEV.jsonl")
DOCCANO_TRAIN_FALLBACK = ""  # auto : merged.jsonl puis batch2.jsonl
MERGE_TRAIN_WITH_FALLBACK = True  # TRAIN.jsonl = 20 CV corrigés → complète avec batch2

# =============================================================================
# SPACY
# =============================================================================

import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans

nlp = spacy.blank("en")

# =============================================================================
# 🎯 LABELS STRICTS AUTORISÉS (SOURCE OF TRUTH)
# =============================================================================

ALLOWED_LABELS = {
    "SKILL",
    "JOB_TITLE",
    "COMPANY",
    "DATE",
    "EXPERIENCE_DESC",
    "DEGREE",
    "INSTITUTION",
    "CERTIFICATION"
}

# alias (pour compatibilité ancienne logique)
TARGET_LABELS = ALLOWED_LABELS

# =============================================================================
# MAPPING DATASET S2
# =============================================================================

LABEL_MAP_D1 = {
    "SKILL": "SKILL",
    "EXPERTISE": "SKILL",

    "DESIGNATION": "JOB_TITLE",

    "COMPANY": "COMPANY",

    "EXPERIENCE": "EXPERIENCE_DESC",

    "EDUCATION": "DEGREE",

    "CERTIFICATION": "CERTIFICATION",

    # ❌ supprimés logiquement
    "ACTION": None,
    "COLLABORATION": None,
    "EMAIL": None,
    "LANGUAGE": None,
    "LOCATION": None,
    "OTHER": None,
    "PERSON": None,
}

# =============================================================================
# MAPPING DATASET S1
# =============================================================================

LABEL_MAP_D2 = {
    "Skills": "SKILL",

    "Designation": "JOB_TITLE",

    "Companies worked at": "COMPANY",

    "Degree": "DEGREE",

    "College Name": "INSTITUTION",

    # ❌ supprimés
    "Name": None,
    "Email Address": None,
    "Location": None,
    "Graduation Year": None,
    "Years of Experience": None,
    "UNKNOWN": None,
}

# =============================================================================
# UTILITAIRES
# =============================================================================

def clean_text(text):
    if not text:
        return ""
    text = str(text)
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
    return normalize_text(text)

def normalize_text(text):
    return " ".join(str(text).split())

def text_fingerprint(text):
    """
    Utilisé pour supprimer les doublons entre datasets
    """
    norm = normalize_text(text).lower()

    return hashlib.sha256(
        norm.encode("utf-8")
    ).hexdigest()

# =============================================================================
# VALIDATION LABEL CENTRALISÉE (IMPORTANT ADDITION)
# =============================================================================

def is_valid_label(label):
    return label in ALLOWED_LABELS

# =============================================================================
# LOG CONFIG
# =============================================================================

print()
print("====================================")
print("CONFIGURATION")
print("====================================")
print("S2_TARGET        :", S2_TARGET)
print("SYNTHETIC_TARGET :", SYNTHETIC_TARGET)
print("S3_WEIGHT        :", S3_WEIGHT)
print("ALLOWED_LABELS   :", len(ALLOWED_LABELS))
print("====================================")

print("OUT  :", OUT)
print("OUT2 :", OUT2)
print("OUT3 :", OUT3)
print()
print("Doccano split fixe :")
print("  TRAIN :", DOCCANO_TRAIN_JSONL)
print("  DEV   :", DOCCANO_DEV_JSONL)


Mounted at /content/drive
✅ Drive monté
PROJECT : /content/drive/MyDrive/cvextraction
BASE    : /content/drive/MyDrive/cvextraction/Datasets

CONFIGURATION
S2_TARGET        : 80
SYNTHETIC_TARGET : 50
S3_WEIGHT        : 1
ALLOWED_LABELS   : 8
OUT  : /content/drive/MyDrive/cvextraction/Datasets/bloc1_processed
OUT2 : /content/drive/MyDrive/cvextraction/Datasets/bloc2_processed
OUT3 : /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed


## SOURCE 1 — 220 CV

In [2]:
#@title 📥 S1 — Resume Entities for NER (Dataturks)

PATH_S1 = os.path.join(
    BASE,
    "Resume Entities for NER",
    "Entity Recognition in Resumes.json"
)

raw_s1 = []

with open(PATH_S1, encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        try:
            raw_s1.append(json.loads(line))
        except Exception:
            continue

print(f"📄 S1 brut : {len(raw_s1)} CV")


# =============================================================================
# Extraction S1
# =============================================================================

def extract_s1(item):

    text = str(item.get("content", "")).strip()

    if not text:
        return "", []

    entities = []

    for ann in item.get("annotation", []):

        labels = ann.get("label", [])
        points = ann.get("points", [])

        if not labels or not points:
            continue

        raw_label = labels[0] if isinstance(labels, list) else labels

        mapped = LABEL_MAP_D2.get(raw_label)

        # ❌ filtre STRICT des labels autorisés
        if mapped is None or mapped not in TARGET_LABELS:
            continue

        for pt in points:

            start = pt.get("start")
            end = pt.get("end")

            if start is None or end is None:
                continue

            try:
                start = int(start)
                end = int(end)
            except:
                continue

            if start < 0 or end <= start or end > len(text):
                continue

            span_text = text[start:end].strip()

            if len(span_text) < 2:
                continue

            entities.append((start, end, mapped))

    entities = sorted(set(entities))

    return text, entities


# =============================================================================
# Construction dataset S1
# =============================================================================

processed_s1 = []
counter_s1 = Counter()
empty_docs = 0

for item in raw_s1:

    text, ents = extract_s1(item)

    if not text or not ents:
        empty_docs += 1
        continue

    processed_s1.append({
        "text": text,
        "entities": ents,
        "source": "S1_DATATURKS"
    })

    for _, _, lbl in ents:
        counter_s1[lbl] += 1


# =============================================================================
# Statistiques
# =============================================================================

print()
print("====================================")
print("S1 NETTOYÉ")
print("====================================")
print("CV retenus      :", len(processed_s1))
print("CV rejetés      :", empty_docs)
print()

print("Distribution labels :")

for lbl, cnt in sorted(counter_s1.items()):
    print(f"{lbl:20s} {cnt}")

print("====================================")

📄 S1 brut : 220 CV

S1 NETTOYÉ
CV retenus      : 220
CV rejetés      : 0

Distribution labels :
COMPANY              728
DEGREE               294
INSTITUTION          330
JOB_TITLE            520
SKILL                467


## SOURCE 2 — 250 CV (filtrés qualité)

Le dataset Kaggle tagge ~83 % des CV **uniquement en SKILL** (bruit : « MARITAL STATUS », « company »…).  
On **rejette** les CV mono-label et on garde les **250 plus diversifiés** (≥ 2 types de labels, SKILL ≤ 85 %).

In [3]:
#@title 📥 S2 — Resume NER Training Dataset (Heuristique)

import re
from collections import Counter

PATH_S2 = os.path.join(
    BASE,
    "Resume NER Training Dataset",
    "train.json"
)

with open(PATH_S2, encoding="utf-8") as f:
    raw_s2 = json.load(f)

print(f"📄 S2 brut : {len(raw_s2)} CV")

# =============================================================================
# PARAMÈTRES QUALITÉ
# =============================================================================

S2_MIN_TEXT_LEN = 250
S2_MIN_ENTITIES = 6
S2_MIN_LABEL_TYPES = 4
S2_MAX_SKILL_RATIO = 0.70


# =============================================================================
# NETTOYAGE UNICODE
# =============================================================================

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)

    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =============================================================================
# EXTRACTION S2 (STRICT TARGET LABELS)
# =============================================================================

def extract_s2(item):

    text = clean_text(
        item.get("text")
        or item.get("content")
        or ""
    )

    if not text:
        return "", []

    entities = []

    doc = nlp.make_doc(text)

    for ann in item.get("annotations", []):

        if not (isinstance(ann, (list, tuple)) and len(ann) == 3):
            continue

        try:
            start = int(ann[0])
            end = int(ann[1])
            raw_label = str(ann[2]).upper()
        except:
            continue

        mapped = LABEL_MAP_D1.get(raw_label)

        # ❌ FILTRE STRICT FINAL
        if mapped is None or mapped not in TARGET_LABELS:
            continue

        if start < 0 or end <= start or end > len(text):
            continue

        span = doc.char_span(
            start,
            end,
            label=mapped,
            alignment_mode="contract"
        )

        if span is None:
            continue

        entities.append(
            (span.start_char, span.end_char, mapped)
        )

    return text, sorted(set(entities))


# =============================================================================
# SCORE QUALITÉ
# =============================================================================

def s2_quality_score(text, entities):

    if len(text) < S2_MIN_TEXT_LEN:
        return -1

    if len(entities) < S2_MIN_ENTITIES:
        return -1

    labels = [lbl for _, _, lbl in entities]
    distinct = set(labels)

    if len(distinct) < S2_MIN_LABEL_TYPES:
        return -1

    skill_ratio = labels.count("SKILL") / len(labels)
    if skill_ratio > S2_MAX_SKILL_RATIO:
        return -1

    score = 0
    score += len(distinct) * 20

    important = {
        "JOB_TITLE",
        "COMPANY",
        "EXPERIENCE_DESC",
        "DEGREE",
        "INSTITUTION",
        "CERTIFICATION"
    }

    score += len(distinct.intersection(important)) * 15
    score += min(len(entities), 40)
    score += min(len(text) // 100, 15)

    return score


# =============================================================================
# FILTRAGE
# =============================================================================

candidates = []
rejected = 0

for item in raw_s2:

    try:
        text, entities = extract_s2(item)
    except:
        continue

    if not text:
        rejected += 1
        continue

    score = s2_quality_score(text, entities)

    if score < 0:
        rejected += 1
        continue

    candidates.append({
        "text": text,
        "entities": entities,
        "source": "S2_HEURISTIC",
        "quality_score": score
    })


# =============================================================================
# TRI + SELECTION
# =============================================================================

candidates.sort(key=lambda x: x["quality_score"], reverse=True)
processed_s2 = candidates[:S2_TARGET]


# =============================================================================
# STATISTIQUES
# =============================================================================

counter_s2 = Counter()

for cv in processed_s2:
    for _, _, lbl in cv["entities"]:
        counter_s2[lbl] += 1

print()
print("====================================")
print("S2 FILTRÉ")
print("====================================")
print("CV bruts            :", len(raw_s2))
print("CV rejetés          :", rejected)
print("CV candidats        :", len(candidates))
print("CV conservés        :", len(processed_s2))
print()

for lbl, cnt in sorted(counter_s2.items()):
    print(f"{lbl:20s} {cnt}")

print("====================================")

📄 S2 brut : 5960 CV

S2 FILTRÉ
CV bruts            : 5960
CV rejetés          : 5485
CV candidats        : 475
CV conservés        : 80

CERTIFICATION        38
COMPANY              15
DEGREE               244
EXPERIENCE_DESC      475
JOB_TITLE            709
SKILL                1391


## SOURCE 3 — Doccano : **+150** nouveaux (objectif **220** uniques S3)

| État | Volume |
|------|--------|
| Déjà annotés (`source3_annotated_merged.jsonl`) | **~80** — exclus auto |
| **Batch 3 (cette cellule)** | **+150** CV ≠ des 80 |
| **Objectif registre S3** | **220** empreintes uniques |

Fichiers : `source3_doccano_import_batch3_150.jsonl` → `source3_doccano_preannot_batch3_150.jsonl`

In [ ]:
#@title S3 — Export +150 CV (objectif 220 uniques, hors ~80 déjà annotés)
import glob
import hashlib

OUT = os.path.join(BASE, 'bloc1_processed')
os.makedirs(OUT, exist_ok=True)

PATH_S3 = os.path.join(BASE, 'Resume_Dataset')
REGISTRY_PATH = os.path.join(OUT, 'source3_export_registry.json')
BATCH3_EXPORT = os.path.join(OUT, 'source3_doccano_import_batch3_150.jsonl')

MERGED_ANNOTATED = os.path.join(OUT, 'source3_annotated_merged.jsonl')

S3_TOTAL_UNIQUE_GOAL = 220    # objectif final CV S3 uniques (registre)
S3_NEW_EXPORT = 150           # nouveaux CV ce lot (≠ des 80 annotés)
S3_BATCH3_SEED = 44
S3_MIN_TEXT_LEN = 150

TEXT_KEYS = ['Text', 'text', 'content', 'resume', 'Resume', 'Resume_str', 'CV', 'Summary']

def get_text(item):
    for k in TEXT_KEYS:
        v = item.get(k) if isinstance(item, dict) else None
        if v and str(v).strip():
            return str(v).strip()
    return ''

def cv_fingerprint(text: str) -> str:
    norm = ' '.join(text.lower().split())[:4000]
    return hashlib.sha256(norm.encode('utf-8')).hexdigest()[:20]

def load_registry():
    if os.path.isfile(REGISTRY_PATH):
        with open(REGISTRY_PATH, encoding='utf-8') as f:
            data = json.load(f)
        return set(data.get('fingerprints', []))
    return set()

def save_registry(fingerprints: set):
    with open(REGISTRY_PATH, 'w', encoding='utf-8') as f:
        json.dump({'fingerprints': sorted(fingerprints)}, f, indent=2)

def register_jsonl(path: str, fps: set) -> int:
    if not os.path.isfile(path):
        return 0
    n = 0
    with open(path, encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            text = item.get('text', '')
            if text:
                fps.add(cv_fingerprint(text))
                n += 1
    return n

if not os.path.isdir(PATH_S3):
    raise FileNotFoundError(f'Dossier introuvable : {PATH_S3}')

# ── Registre : CV déjà vus (annotés + anciens imports) ──
used_fps = load_registry()
exclude_paths = list(dict.fromkeys(
    [MERGED_ANNOTATED]
    + sorted(glob.glob(os.path.join(OUT, 'source3_doccano_import*.jsonl')))
    + sorted(glob.glob(os.path.join(OUT, 'source3_annotated*.jsonl')))
    + sorted(glob.glob(os.path.join(OUT, 'source3_doccano_preannot*.jsonl')))
    + [BATCH3_EXPORT]
))

counts = {}
for p in exclude_paths:
    counts[os.path.basename(p)] = register_jsonl(p, used_fps)

S3_ALREADY_DONE = counts.get('source3_annotated_merged.jsonl', 0)
if S3_ALREADY_DONE == 0 and os.path.isfile(MERGED_ANNOTATED):
    raise FileNotFoundError('source3_annotated_merged.jsonl introuvable sur Drive')

slots_left = max(0, S3_TOTAL_UNIQUE_GOAL - len(used_fps))
S3_NEW_SIZE = min(S3_NEW_EXPORT, slots_left)
if S3_NEW_SIZE < S3_NEW_EXPORT:
    print(f'⚠️ Plafond {S3_TOTAL_UNIQUE_GOAL} : export {S3_NEW_SIZE} CV (au lieu de {S3_NEW_EXPORT})')

save_registry(used_fps)
print(f'Annotés exclus (merged) : {S3_ALREADY_DONE} CV')
print(f'Réserve anti-doublon : {len(used_fps)} empreintes')
print('Fichiers exclus :', {k: v for k, v in counts.items() if v > 0})
print(f'Ce lot : +{S3_NEW_SIZE} nouveaux → registre ~{len(used_fps) + S3_NEW_SIZE} (cible {S3_TOTAL_UNIQUE_GOAL})')

# ── Pool Resume_Dataset ──
candidates = []
for fp in glob.glob(os.path.join(PATH_S3, '*.jsonl')):
    with open(fp, encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            text = get_text(json.loads(line))
            if len(text) < S3_MIN_TEXT_LEN:
                continue
            fp_id = cv_fingerprint(text)
            if fp_id in used_fps:
                continue
            candidates.append((fp_id, text))

# Dédupliquer le pool (même texte 2× dans le fichier source)
seen = {}
for fp_id, text in candidates:
    seen.setdefault(fp_id, text)
unique = list(seen.items())
print(f'Pool disponible (hors lot 1) : {len(unique)} CV')

if len(unique) < S3_NEW_SIZE:
    raise ValueError(
        f'Pas assez de CV uniques : {len(unique)} < {S3_NEW_SIZE}. '
        'Vérifie le registre ou réduis S3_NEW_SIZE.'
    )

rng = random.Random(S3_BATCH3_SEED)
picked = rng.sample(unique, S3_NEW_SIZE)

with open(BATCH3_EXPORT, 'w', encoding='utf-8') as f:
    for fp_id, text in picked:
        f.write(json.dumps({'text': text[:5000], 'label': []}, ensure_ascii=False) + '\n')
        used_fps.add(fp_id)

# Vérif : aucun des 150 n'est dans les 80 annotés
merged_fps = set()
register_jsonl(MERGED_ANNOTATED, merged_fps)
overlap = sum(1 for fp_id, _ in picked if fp_id in merged_fps)
assert overlap == 0, f'ERREUR : {overlap} doublons avec merged — ne pas exporter'

save_registry(used_fps)
print('Export lot 3 :', BATCH3_EXPORT)
print(f'{S3_NEW_SIZE} CV nouveaux (0 chevauchement avec les {S3_ALREADY_DONE} annotés)')
print(f'Total registre unique : {len(used_fps)} (objectif {S3_TOTAL_UNIQUE_GOAL})')
print('→ Pré-annotation batch3 puis Doccano')


Réserve anti-doublon : 165 empreintes
  lot1 import: 50 | annotés lot1: 35 | batch2 déjà exporté: 0
Pool disponible (hors lot 1) : 3138 CV
Export lot 2 : /content/drive/MyDrive/cvextraction/Datasets/bloc1_processed/source3_doccano_import_batch2.jsonl
115 CV — aucun doublon avec les 165 déjà réservés
Prochaine étape : cellule pré-annotation puis import Doccano


In [ ]:
#@title S3 — Pré-annotation lot 3 (+150 CV)
BATCH3_PREANNOT = os.path.join(OUT, 'source3_doccano_preannot_batch3_150.jsonl')
TRF_RUN = os.path.join(BASE, 'models', 'cv_ner_trf_run')

def is_valid_spacy_model(path: str) -> bool:
    return os.path.isdir(path) and os.path.isfile(os.path.join(path, 'meta.json'))

def is_transformer_model(path: str) -> bool:
    cfg = os.path.join(path, 'config.cfg')
    if not os.path.isfile(cfg):
        return False
    with open(cfg, encoding='utf-8') as f:
        return 'transformer' in f.read().lower()

TRF_CANDIDATES = [
    os.path.join(TRF_RUN, 'model-best'),
    os.path.join(BASE, 'models', 'cv_ner_trf_best'),
    os.path.join(TRF_RUN, 'model-last'),
]
FALLBACK = os.path.join(BASE, 'models', 'cv_ner_best')

if not os.path.isfile(BATCH3_EXPORT):
    raise FileNotFoundError(f'Exporte d\'abord le lot 3 : {BATCH3_EXPORT}')

trf_path = next((p for p in TRF_CANDIDATES if is_valid_spacy_model(p)), None)
use_trf = trf_path is not None and is_transformer_model(trf_path)
nlp_s3 = None
model_path = None

# ── Tentative RoBERTa (versions alignées Colab) ─────────────────────────────
if use_trf:
    print('Installation stack transformer (numpy 1.26 + scipy 1.11)...')
    get_ipython().system('pip install -q "numpy==1.26.4" "scipy==1.11.4" "scikit-learn==1.3.2"')
    get_ipython().system('pip install -q "transformers==4.36.2" "spacy==3.7.5" "spacy-transformers==1.3.5" spacy-lookups-data')
    try:
        import spacy_transformers  # enregistre factory "transformer"
        import spacy
        try:
            spacy.prefer_gpu()
        except Exception:
            pass
        nlp_s3 = spacy.load(trf_path)
        model_path = trf_path
        print('Modèle RoBERTa chargé :', model_path)
        print('Pipeline :', nlp_s3.pipe_names)
    except Exception as e:
        print('⚠️ RoBERTa non chargé :', e)
        print('→ Bascule sur cv_ner_best (blank)')

# ── Secours : modèle blank (déjà fonctionnel chez toi) ───────────────────────
if nlp_s3 is None:
    if not is_valid_spacy_model(FALLBACK):
        raise FileNotFoundError(
            f'Aucun modèle utilisable.\n'
            f'RoBERTa : {trf_path}\n'
            f'Secours : {FALLBACK}'
        )
    get_ipython().system('pip install -q spacy==3.7.5 spacy-lookups-data')
    import spacy
    nlp_s3 = spacy.load(FALLBACK)
    model_path = FALLBACK
    print('Modèle chargé (secours blank, F1~0.70) :', model_path)
    print('Suggestions moins bonnes → plus de corrections dans Doccano')

# ── Pré-annotation ───────────────────────────────────────────────────────────
n_spans = 0
with open(BATCH3_EXPORT, encoding='utf-8') as fin, \
     open(BATCH3_PREANNOT, 'w', encoding='utf-8') as fout:
    for line in fin:
        item = json.loads(line)
        text = item['text']
        doc = nlp_s3(text[:1_000_000])
        labels = [
            [int(ent.start_char), int(ent.end_char), ent.label_]
            for ent in doc.ents
            if ent.label_ in TARGET_LABELS
        ]
        n_spans += len(labels)
        fout.write(json.dumps({'text': text, 'label': labels}, ensure_ascii=False) + '\n')

print('Pré-annotation :', BATCH3_PREANNOT)
print(f'{S3_NEW_SIZE} CV | {n_spans} spans proposés')
print('Doccano : importer ce fichier (labels = suggestions à corriger)')

Installation stack transformer (numpy 1.26 + scipy 1.11)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 93.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.3.2 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.11.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires scipy>=1.13, but you have scipy 1.11.4 which is incompatible.
inequality 1.1.2 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
imbalanced-learn 0.14.1 requires scikit-learn<2,>=1.4.2, but you have scikit

/usr/local/lib/python3.12/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_pipeline' (0.0.0) was trained with spaCy v3.7.5 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


⚠️ RoBERTa non chargé : No module named 'transformers.models.deprecated.mega'
→ Bascule sur cv_ner_best (blank)
Modèle chargé (secours blank, F1~0.70) : /content/drive/MyDrive/cvextraction/Datasets/models/cv_ner_best
Suggestions moins bonnes → plus de corrections dans Doccano
Pré-annotation : /content/drive/MyDrive/cvextraction/Datasets/bloc1_processed/source3_doccano_preannot_batch2.jsonl
115 CV | 1526 spans proposés
Doccano : importer ce fichier (labels = suggestions à corriger)


In [ ]:
import os

for candidate in MODEL_CANDIDATES:
    print(f"\n📁 {candidate}")
    print(f"   exists: {os.path.isdir(candidate)}")
    if os.path.isdir(candidate):
        files = os.listdir(candidate)
        print(f"   contents: {files}")


📁 /content/drive/MyDrive/cvextraction/Datasets/models/cv_ner_trf_run/model-best
   exists: True
   contents: ['transformer', 'vocab', 'ner']

📁 /content/drive/MyDrive/cvextraction/Datasets/models/cv_ner_trf_best
   exists: False

📁 /content/drive/MyDrive/cvextraction/Datasets/models/cv_ner_best
   exists: True
   contents: ['ner', 'vocab', 'tokenizer', 'config.cfg', 'meta.json']


## Export ~400 CV → Doccano (correction complète)

Cellule **Export ~400 CV → JSONL Doccano** → télécharge `doccano_import_ALL_400.jsonl`

**Doccano :** nouveau projet **Sequence Labeling** → 6 labels → Import JSONL

In [ ]:
#@title 📤 Export ~400 CV → JSONL Doccano (correction complète)

import glob
import json
import os
import re
import unicodedata
from collections import Counter

if "OUT" not in globals():
    PROJECT = "/content/drive/MyDrive/cvextraction"
    BASE = os.path.join(PROJECT, "Datasets")
    OUT = os.path.join(BASE, "bloc1_processed")
    OUT3 = os.path.join(BASE, "bloc3_processed")

LABELS_6 = {
    "SKILL", "JOB_TITLE", "COMPANY", "DATE", "DEGREE", "INSTITUTION",
}
OUTPUT = os.path.join(OUT, "doccano_import_ALL_400.jsonl")


def _norm_text(text):
    if not text:
        return ""
    text = str(text)
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _entities_from_item(item, text):
    out = []
    raw = item.get("label") or item.get("labels") or item.get("entities") or []
    if isinstance(raw, dict):
        raw = raw.get("entities", [])
    for ann in raw:
        if isinstance(ann, (list, tuple)) and len(ann) >= 3:
            start, end, label = int(ann[0]), int(ann[1]), str(ann[2]).strip().upper()
        elif isinstance(ann, dict):
            start = int(ann.get("start_offset", ann.get("start", 0)))
            end = int(ann.get("end_offset", ann.get("end", 0)))
            label = str(ann.get("label", "")).strip().upper()
        else:
            continue
        if label not in LABELS_6:
            continue
        if start < 0 or end <= start or end > len(text):
            continue
        if len(text[start:end].strip()) < 2:
            continue
        out.append([start, end, label])
    return sorted(set(tuple(x) for x in out))


def _to_doccano_line(text, entities, meta=None):
    row = {"text": text, "label": [[s, e, l] for s, e, l in entities]}
    if meta:
        row["meta"] = meta
    return row


def _find_source():
    candidates = [
        os.path.join(OUT, "source3_annotated_merged.jsonl"),
        os.path.join(OUT, "source3_annotated_batch2.jsonl"),
        os.path.join(OUT3, "bloc3_natural.json"),
    ]
    candidates += sorted(glob.glob(os.path.join(OUT, "source3_annotated*.jsonl")))
    seen = set()
    for p in candidates:
        if not p or p in seen or not os.path.isfile(p):
            continue
        seen.add(p)
        if p.endswith(".json"):
            with open(p, encoding="utf-8") as f:
                n = len(json.load(f))
        else:
            n = sum(1 for _ in open(p, encoding="utf-8"))
        if n >= 300:
            return p, n
    return None, 0


def _load_records(source_path):
    records = []
    if source_path.endswith(".json"):
        with open(source_path, encoding="utf-8") as f:
            raw = json.load(f)
        for item in raw:
            text = _norm_text(item.get("text", ""))
            if not text:
                continue
            ents = item.get("entities") or []
            parsed = []
            for ann in ents:
                if isinstance(ann, (list, tuple)) and len(ann) >= 3:
                    s, e, l = int(ann[0]), int(ann[1]), str(ann[2]).strip().upper()
                    if l in LABELS_6 and 0 <= s < e <= len(text):
                        parsed.append((s, e, l))
            records.append({"text": text, "entities": sorted(set(parsed))})
        return records
    with open(source_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = _norm_text(item.get("text") or item.get("data") or "")
            if not text:
                continue
            entities = _entities_from_item(item, text)
            records.append({"text": text, "entities": entities})
    return records


source_path, source_n = _find_source()
if not source_path:
    raise FileNotFoundError(
        "Aucune source ~400 CV trouvée.\n"
        f"Attendu : {OUT}/source3_annotated_batch2.jsonl"
    )

print("Source :", source_path, f"({source_n} entrées)")
records = _load_records(source_path)
print("CV chargés :", len(records))

# Dédup par texte
seen = set()
unique = []
for cv in records:
    fp = hash(_norm_text(cv["text"]).lower())
    if fp in seen:
        continue
    seen.add(fp)
    unique.append(cv)
records = unique
print("CV uniques :", len(records))

label_counts = Counter()
empty = 0
with open(OUTPUT, "w", encoding="utf-8") as out:
    for i, cv in enumerate(records, start=1):
        text = cv["text"]
        entities = cv.get("entities") or []
        if not entities:
            empty += 1
        for _, _, l in entities:
            label_counts[l] += 1
        hint = text[:140].replace("\n", " ").strip()
        line = _to_doccano_line(
            text,
            entities,
            meta={"id": i, "search_hint": hint},
        )
        out.write(json.dumps(line, ensure_ascii=False) + "\n")

print()
print("=" * 50)
print("EXPORT DOCCANO")
print("=" * 50)
print("Fichier :", OUTPUT)
print("CV exportés :", len(records))
print("CV sans label 6L :", empty, "(à annoter dans Doccano)")
print("Labels :")
for lbl, n in sorted(label_counts.items()):
    print(f"  {lbl:15s} {n}")

print()
print("IMPORT DOCCANO")
print("-" * 50)
print("1. Créer projet : Sequence Labeling")
print("2. Labels : SKILL, JOB_TITLE, COMPANY, DATE, DEGREE, INSTITUTION")
print("3. Dataset → Import →", os.path.basename(OUTPUT))
print("4. Corriger → Export JSONL → source3_annotated_merged.jsonl sur Drive")

try:
    from google.colab import files
    print()
    print("Téléchargement local...")
    files.download(OUTPUT)
except ImportError:
    print("(Hors Colab : récupère le fichier sur Drive)")


In [ ]:
#@title 📤 Upload TRAIN.jsonl + DEV.jsonl → Drive (après correction Doccano)

import os
import shutil

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DEST = OUT if "OUT" in globals() else "/content/drive/MyDrive/cvextraction/Datasets/bloc1_processed"
os.makedirs(DEST, exist_ok=True)

print("Destination Drive :", DEST)
print()

if IN_COLAB:
    print("1) Clique « Choose Files » et sélectionne TRAIN.jsonl puis DEV.jsonl")
    uploaded = files.upload()  # noqa: F821 — google.colab.files
    for name, data in uploaded.items():
        low = name.lower()
        if low == "train.jsonl":
            target = os.path.join(DEST, "TRAIN.jsonl")
        elif low == "dev.jsonl":
            target = os.path.join(DEST, "DEV.jsonl")
        else:
            target = os.path.join(DEST, name)
        with open(target, "wb") as f:
            f.write(data)
        n = sum(1 for _ in open(target, encoding="utf-8"))
        print(f"✅ {name} → {target} ({n} lignes)")
else:
    print("Hors Colab : copie manuellement TRAIN.jsonl et DEV.jsonl vers :")
    print(DEST)

print()
print("=== Vérification ===")
for fname in ("TRAIN.jsonl", "DEV.jsonl"):
    p = os.path.join(DEST, fname)
    if os.path.isfile(p):
        n = sum(1 for _ in open(p, encoding="utf-8"))
        print(f"✅ {fname} : {n} CV — {p}")
    else:
        print(f"❌ MANQUANT : {p}")

if os.path.isfile(os.path.join(DEST, "TRAIN.jsonl")) and os.path.isfile(os.path.join(DEST, "DEV.jsonl")):
    print()
    print("→ Exécute maintenant « S3 — Import Doccano » puis « Split train/dev »")
else:
    print()
    print("⚠️ Les 2 fichiers sont requis avant de continuer.")


## SOURCE 3 — Import Doccano

**Priorité 1 — export complet ~400 CV + DEV fixe** :
- `source3_annotated_merged.jsonl` (~400 corrigés) + `DEV.jsonl` (60)
- Supprime `TRAIN.jsonl` (20 partiels)

**Priorité 2 — partiel** :
- `TRAIN.jsonl` (20) + `DEV.jsonl` (60) + fallback batch2

**Priorité 3 — legacy** :
- `source3_annotated_batch2.jsonl` seul → split aléatoire

Format : `{"text":"...", "label": [[start, end, "SKILL"], ...]}`

In [4]:
#@title 📥 S3 — Import Doccano (TRAIN.jsonl + DEV.jsonl ou merged)

import glob
import json
import re
import unicodedata
from collections import Counter

MERGED_ANNOTATED = os.path.join(OUT, "source3_annotated_merged.jsonl")

PRE_SPLIT_TRAIN = None
PRE_SPLIT_DEV = None
USE_FIXED_DOCCANNO_SPLIT = False


def _resolve_dev_path():
    for p in (
        DOCCANO_DEV_JSONL,
        os.path.join(OUT3, "DEV.jsonl"),
        os.path.join(OUT, "dev.jsonl"),
    ):
        if p and os.path.isfile(p):
            return p
    return None


def _resolve_train_path():
    for p in (
        DOCCANO_TRAIN_JSONL,
        os.path.join(OUT3, "TRAIN.jsonl"),
        os.path.join(OUT, "train.jsonl"),
    ):
        if p and os.path.isfile(p):
            return p
    return None


def _resolve_doccano_split_paths():
    dev_p = _resolve_dev_path()
    train_p = _resolve_train_path()
    if dev_p and train_p:
        return train_p, dev_p
    return None, None


def _resolve_full_corrected_path(min_cv=350):
    for p in (
        os.path.join(OUT, "source3_annotated_merged.jsonl"),
        os.path.join(OUT, "source3_annotated_batch2.jsonl"),
        os.path.join(OUT, "CORRECTED_400.jsonl"),
    ):
        if os.path.isfile(p):
            n = sum(1 for _ in open(p, encoding="utf-8"))
            if n >= min_cv:
                return p, n
    return None, 0


def _split_full_plus_dev(full_records, dev_records):
    dev_fps = {text_fingerprint(cv["text"]) for cv in dev_records}
    full_by_fp = {text_fingerprint(cv["text"]): cv for cv in full_records}
    dev_out = []
    for cv in dev_records:
        fp = text_fingerprint(cv["text"])
        dev_out.append(full_by_fp.get(fp, cv))
    train_out = [cv for cv in full_records if text_fingerprint(cv["text"]) not in dev_fps]
    return train_out, dev_out


def _resolve_train_fallback_path():
    if DOCCANO_TRAIN_FALLBACK and os.path.isfile(DOCCANO_TRAIN_FALLBACK):
        return DOCCANO_TRAIN_FALLBACK
    for p in (
        os.path.join(OUT, "source3_annotated_merged.jsonl"),
        os.path.join(OUT, "source3_annotated_batch2.jsonl"),
    ):
        if os.path.isfile(p):
            return p
    return None


def _print_split_diagnostics():
    print("⚠️ TRAIN.jsonl + DEV.jsonl introuvables — mode legacy activé.")
    print("   Chemins vérifiés :")
    for p in (DOCCANO_TRAIN_JSONL, DOCCANO_DEV_JSONL):
        print(f"     {'✅' if os.path.isfile(p) else '❌'} {p}")
    found = glob.glob(os.path.join(BASE, "**", "*.jsonl"), recursive=True)
    hits = [p for p in found if os.path.basename(p).lower() in ("train.jsonl", "dev.jsonl")]
    if hits:
        print("   Fichiers train/dev trouvés ailleurs (déplace-les vers bloc1_processed/) :")
        for p in sorted(hits):
            print(f"     → {p}")
    else:
        print("   Aucun train.jsonl / dev.jsonl sous Datasets/")
    print("   → Uploade sur Drive : bloc1_processed/TRAIN.jsonl + DEV.jsonl")


def clean_text_doccano(text):
    if not text:
        return ""
    text = str(text)
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_label(label):
    if not label:
        return None
    label = str(label).strip().upper().replace(" ", "_")
    return label


def parse_doccano(item):
    text = clean_text_doccano(item.get("text") or item.get("data") or "")
    if not text:
        return [], text

    entities = []
    raw = item.get("labels") or item.get("label") or []
    if isinstance(raw, dict):
        raw = raw.get("entities", [])

    for ann in raw:
        if isinstance(ann, (list, tuple)) and len(ann) == 3:
            try:
                start, end, label = int(ann[0]), int(ann[1]), normalize_label(ann[2])
            except Exception:
                continue
        elif isinstance(ann, dict):
            try:
                start = int(ann.get("start_offset", ann.get("start")))
                end = int(ann.get("end_offset", ann.get("end")))
                label = normalize_label(ann.get("label", ""))
            except Exception:
                continue
        else:
            continue

        if label not in TARGET_LABELS:
            continue
        if start < 0 or end <= start or end > len(text):
            continue
        if len(text[start:end].strip()) < 2:
            continue
        entities.append((start, end, label))

    return sorted(set(entities)), text


def load_doccano_jsonl(path, split_tag="S3_DOCCANO"):
    records = []
    rejected = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                rejected += 1
                continue
            entities, text = parse_doccano(item)
            if not text or not entities:
                rejected += 1
                continue
            records.append({
                "text": text,
                "entities": entities,
                "source": split_tag,
            })
    return records, rejected


def merge_train_with_fallback(corrected_train, fallback_path):
    if not os.path.isfile(fallback_path):
        return corrected_train

    fallback_train, _ = load_doccano_jsonl(fallback_path, "S3_DOCCANO")
    corrected_by_fp = {text_fingerprint(cv["text"]): cv for cv in corrected_train}
    merged = []
    used_corrected = set()

    for cv in fallback_train:
        fp = text_fingerprint(cv["text"])
        if fp in corrected_by_fp:
            merged.append(corrected_by_fp[fp])
            used_corrected.add(fp)
        else:
            merged.append(cv)

    for cv in corrected_train:
        fp = text_fingerprint(cv["text"])
        if fp not in used_corrected:
            merged.append(cv)

    return merged


def collect_annotated_files():
    if os.path.isfile(MERGED_ANNOTATED):
        return [MERGED_ANNOTATED]
    patterns = ["source3_annotated*.jsonl", "doccano*.jsonl", "*annotated*.jsonl"]
    files = []
    for pattern in patterns:
        files.extend(glob.glob(os.path.join(OUT, pattern)))
    skip = {"TRAIN.jsonl", "DEV.jsonl", "train.jsonl", "dev.jsonl"}
    return sorted({f for f in files if os.path.basename(f) not in skip})


processed_s3 = []
counter_s3 = Counter()
train_path, dev_path = _resolve_doccano_split_paths()
dev_only_path = _resolve_dev_path()
full_path, full_n = _resolve_full_corrected_path()

if dev_only_path and full_path and (not train_path or full_n >= 350):
    USE_FIXED_DOCCANNO_SPLIT = True
    print("Mode : FULL CORRIGÉ (~400) + DEV fixe")
    print("  FULL :", full_path, f"({full_n} CV)")
    print("  DEV  :", dev_only_path)

    all_records, full_rejected = load_doccano_jsonl(full_path, "S3_DOCCANO_FULL")
    PRE_SPLIT_DEV, dev_rejected = load_doccano_jsonl(dev_only_path, "S3_DOCCANO_DEV")
    PRE_SPLIT_TRAIN, PRE_SPLIT_DEV = _split_full_plus_dev(all_records, PRE_SPLIT_DEV)

    processed_s3 = PRE_SPLIT_TRAIN + PRE_SPLIT_DEV
    print(f"  FULL chargé  : {len(all_records)} CV (rejetés {full_rejected})")
    print(f"  → train      : {len(PRE_SPLIT_TRAIN)} CV (400 − dev)")
    print(f"  → dev        : {len(PRE_SPLIT_DEV)} CV (rejetés {dev_rejected})")

elif train_path and dev_path:
    USE_FIXED_DOCCANNO_SPLIT = True
    print("Mode : SPLIT FIXE Doccano")
    print("  TRAIN :", train_path)
    print("  DEV   :", dev_path)

    PRE_SPLIT_DEV, dev_rejected = load_doccano_jsonl(dev_path, "S3_DOCCANO_DEV")
    corrected_train, train_rejected = load_doccano_jsonl(train_path, "S3_DOCCANO_TRAIN")

    if MERGE_TRAIN_WITH_FALLBACK and len(corrected_train) < 100:
        fallback_path = _resolve_train_fallback_path()
        if not fallback_path:
            print("  ⚠️ Pas de fallback train — seulement", len(corrected_train), "CV train")
            PRE_SPLIT_TRAIN = corrected_train
        else:
            PRE_SPLIT_TRAIN = merge_train_with_fallback(corrected_train, fallback_path)
            print(
                f"  Train corrigés : {len(corrected_train)} → fusion {os.path.basename(fallback_path)}"
                f" : {len(PRE_SPLIT_TRAIN)}"
            )
    else:
        PRE_SPLIT_TRAIN = corrected_train

    dev_fps = {text_fingerprint(cv["text"]) for cv in PRE_SPLIT_DEV}
    overlap = sum(1 for cv in PRE_SPLIT_TRAIN if text_fingerprint(cv["text"]) in dev_fps)
    if overlap:
        PRE_SPLIT_TRAIN = [cv for cv in PRE_SPLIT_TRAIN if text_fingerprint(cv["text"]) not in dev_fps]
        print(f"  ⚠️ {overlap} CV retirés du train (présents dans dev)")

    processed_s3 = PRE_SPLIT_TRAIN + PRE_SPLIT_DEV
    print(f"  DEV chargé   : {len(PRE_SPLIT_DEV)} CV (rejetés {dev_rejected})")
    print(f"  TRAIN chargé : {len(PRE_SPLIT_TRAIN)} CV (rejetés {train_rejected})")

    # Cache pour la cellule Split (survit au redémarrage kernel)
    for fname, data in (
        ("doccano_fixed_split_train.json", PRE_SPLIT_TRAIN),
        ("doccano_fixed_split_dev.json", PRE_SPLIT_DEV),
    ):
        with open(os.path.join(OUT3, fname), "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
    print("  Cache split →", OUT3)

else:
    _print_split_diagnostics()
    print()
    annotated_files = collect_annotated_files()
    if not annotated_files:
        raise FileNotFoundError(
            "Aucun export Doccano trouvé.\n"
            f"Attendu : {DOCCANO_TRAIN_JSONL} + {DOCCANO_DEV_JSONL}\n"
            f"     ou : {MERGED_ANNOTATED}"
        )
    print("Mode : merged / legacy")
    print("Fichiers détectés :")
    seen_docs = set()
    duplicate_docs = empty_docs = 0
    for file_path in annotated_files:
        print(" -", os.path.basename(file_path))
        imported = 0
        with open(file_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    continue
                entities, text = parse_doccano(item)
                if not text or not entities:
                    empty_docs += 1
                    continue
                fp = text_fingerprint(text)
                if fp in seen_docs:
                    duplicate_docs += 1
                    continue
                seen_docs.add(fp)
                processed_s3.append({"text": text, "entities": entities, "source": "S3_DOCCANO"})
                imported += 1
                for _, _, lbl in entities:
                    counter_s3[lbl] += 1
        print(f"{os.path.basename(file_path)} → {imported} CV")

for cv in processed_s3:
    for _, _, lbl in cv["entities"]:
        counter_s3[lbl] += 1

print()
print("====================================")
print("S3 DOCCANO")
print("====================================")
print("Split fixe         :", USE_FIXED_DOCCANNO_SPLIT)
print("CV retenus         :", len(processed_s3))
if USE_FIXED_DOCCANNO_SPLIT:
    print("  → train          :", len(PRE_SPLIT_TRAIN))
    print("  → dev            :", len(PRE_SPLIT_DEV))
print()
print("Distribution labels :")
for lbl, cnt in sorted(counter_s3.items()):
    print(f"{lbl:20s} {cnt}")
print("====================================")


Fichiers détectés :
 - source3_annotated_batch2.jsonl
source3_annotated_batch2.jsonl → 398 CV

S3 DOCCANO
CV retenus         : 398
Doublons supprimés : 2
Rejetés            : 0

Distribution labels :
CERTIFICATION        197
COMPANY              992
DATE                 1421
DEGREE               383
EXPERIENCE_DESC      1044
INSTITUTION          377
JOB_TITLE            1183
SKILL                5190


In [5]:
#@title 🔀 Fusion Finale S1 + S2 + S3 (pondérée) — optionnel si doccano_only

from collections import Counter

# doccano_only + TRAIN.jsonl/DEV.jsonl → cette cellule n'est pas nécessaire
_skip_bloc1_fusion = (
    globals().get("USE_FIXED_DOCCANNO_SPLIT")
    or TRAIN_MODE == "doccano_only"
)

if _skip_bloc1_fusion:
    print("⏭️ Mode doccano_only / split fixe — fusion S1+S2+S3 ignorée.")
    print(f"   processed_s3 : {len(processed_s3)} CV")
    print("   → Passe à la cellule « Bloc3 — fusion F1-MAX »")
    bloc1_all = []
else:

    if "processed_s1" not in globals() or "processed_s2" not in globals():
        raise RuntimeError(
            "processed_s1 / processed_s2 introuvables.\n"
            "Exécute les cellules S1 et S2, ou mets TRAIN_MODE = 'doccano_only'."
        )

    # =============================================================================
    # PONDÉRATION
    # =============================================================================

    print("📊 Avant pondération")
    print("S1 :", len(processed_s1))
    print("S2 :", len(processed_s2))
    print("S3 :", len(processed_s3))
    print()

    weighted_s3 = processed_s3 * S3_WEIGHT

    print("📊 Après pondération")
    print("S1 :", len(processed_s1))
    print("S2 :", len(processed_s2))
    print("S3 :", len(weighted_s3))
    print()

    # =============================================================================
    # FUSION
    # =============================================================================

    merged = processed_s1 + processed_s2 + weighted_s3

    print("Total avant déduplication :", len(merged))


    # =============================================================================
    # CLEAN + FILTER STRICT LABELS (IMPORTANT FIX)
    # =============================================================================

    ALLOWED = TARGET_LABELS  # sécurité explicite

    def clean_entities(entities):

        cleaned = []

        for s, e, lbl in entities:

            try:
                s = int(s)
                e = int(e)
                lbl = str(lbl).strip().upper()
            except:
                continue

            # ❌ FILTRE FINAL STRICT
            if lbl not in ALLOWED:
                continue

            if e <= s:
                continue

            cleaned.append((s, e, lbl))

        # dédup stable
        seen_local = set()
        ordered = []

        for ent in cleaned:
            if ent not in seen_local:
                ordered.append(ent)
                seen_local.add(ent)

        return ordered


    # =============================================================================
    # DÉDUP INTER-SOURCES
    # =============================================================================

    by_fp = {}
    duplicates_removed = 0
    SOURCE_RANK = {"S3_DOCCANO": 3, "S1_DATATURKS": 2, "S2_HEURISTIC": 1, "SYNTHETIC": 0}

    for cv in merged:
        fp = text_fingerprint(cv["text"])
        entities = clean_entities(cv["entities"])

        # ❌ sécurité supplémentaire : skip CV vide après nettoyage
        if not entities:
            continue

        record = {
            "text": cv["text"],
            "entities": entities,
            "source": cv["source"]
        }

        prev = by_fp.get(fp)
        if prev is None:
            by_fp[fp] = record
        else:
            duplicates_removed += 1
            rank_new = SOURCE_RANK.get(record["source"], 0)
            rank_old = SOURCE_RANK.get(prev["source"], 0)
            if rank_new >= rank_old:
                by_fp[fp] = record

    bloc1_all = list(by_fp.values())


    print("Doublons supprimés :", duplicates_removed)
    print("Total final        :", len(bloc1_all))


    # =============================================================================
    # STATS PAR SOURCE
    # =============================================================================

    source_counter = Counter()

    for cv in bloc1_all:
        source_counter[cv["source"]] += 1

    print()
    print("====================================")
    print("RÉPARTITION PAR SOURCE")
    print("====================================")

    for src, cnt in sorted(source_counter.items()):
        print(f"{src:20s} {cnt}")


    # =============================================================================
    # STATS LABELS
    # =============================================================================

    label_counter = Counter()

    for cv in bloc1_all:
        for _, _, lbl in cv["entities"]:
            label_counter[lbl] += 1

    print()
    print("====================================")
    print("RÉPARTITION DES LABELS")
    print("====================================")

    for lbl, cnt in sorted(label_counter.items()):
        print(f"{lbl:20s} {cnt}")


    # =============================================================================
    # LONGUEUR DES CV
    # =============================================================================

    if bloc1_all:

        avg_chars = int(sum(len(cv["text"]) for cv in bloc1_all) / len(bloc1_all))
        avg_words = int(sum(len(cv["text"].split()) for cv in bloc1_all) / len(bloc1_all))
    else:
        avg_chars = 0
        avg_words = 0

    print()
    print("====================================")
    print("LONGUEUR MOYENNE")
    print("====================================")
    print("Caractères :", avg_chars)
    print("Mots       :", avg_words)


    # =============================================================================
    # SAUVEGARDE
    # =============================================================================

    bloc1_path = os.path.join(OUT, "bloc1_all.json")

    with open(bloc1_path, "w", encoding="utf-8") as f:
        json.dump(bloc1_all, f, ensure_ascii=False)

    print()
    print("💾 Dataset sauvegardé :")
    print(bloc1_path)


    # =============================================================================
    # CONTRÔLES QUALITÉ
    # =============================================================================

    assert len(bloc1_all) > 0

    assert len({
        text_fingerprint(cv["text"]) for cv in bloc1_all
    }) == len(bloc1_all)

    print()
    print("✅ Contrôles qualité OK")


📊 Avant pondération
S1 : 220
S2 : 80
S3 : 398

📊 Après pondération
S1 : 220
S2 : 80
S3 : 398

Total avant déduplication : 698
Doublons supprimés : 11
Total final        : 687

RÉPARTITION PAR SOURCE
S1_DATATURKS         219
S2_HEURISTIC         70
S3_DOCCANO           398

RÉPARTITION DES LABELS
CERTIFICATION        235
COMPANY              1722
DATE                 1421
DEGREE               900
EXPERIENCE_DESC      1509
INSTITUTION          706
JOB_TITLE            2393
SKILL                7030

LONGUEUR MOYENNE
Caractères : 2886
Mots       : 392

💾 Dataset sauvegardé :
/content/drive/MyDrive/cvextraction/Datasets/bloc1_processed/bloc1_all.json

✅ Contrôles qualité OK


## Fusion Bloc 1 + export spaCy

In [6]:
#@title 🤖 Génération Synthetic Dataset (50 CV max)

import os
import json
import random
import re

RNG = random.Random(SEED)
TARGET = SYNTHETIC_TARGET

os.makedirs(OUT2, exist_ok=True)

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
FIRST_NAMES = ["Alex", "Sam", "Jordan", "Taylor", "Morgan", "Casey", "Riley", "Avery"]
LAST_NAMES = ["Martin", "Bernard", "Dubois", "Smith", "Johnson", "Brown", "Garcia", "Miller"]
JOB_TITLES = [
    "Software Engineer", "Data Analyst", "Project Manager", "Business Analyst",
    "Full Stack Developer", "DevOps Engineer", "Product Owner", "Consultant",
]
COMPANIES = ["Acme Corp", "Global Solutions", "Tech Partners", "Innovate Ltd", "NextGen Systems"]
DEGREES = ["Bachelor of Science", "Master of Engineering", "MBA", "Licence Informatique", "Master Data Science"]
INSTITUTIONS = ["State University", "Engineering School", "Business School", "National Institute"]
SKILLS = [
    "Python", "Java", "SQL", "Git", "Linux", "Agile", "REST APIs",
    "Docker", "Cloud", "Testing", "Communication", "Team leadership",
]
EXPERIENCE_DESC = [
    "Led cross-functional projects and delivered measurable improvements.",
    "Developed and maintained business applications in production.",
    "Collaborated with stakeholders to define requirements and timelines.",
    "Optimized processes and documented technical solutions.",
]
CERTIFICATIONS = ["PMP", "AWS Certified", "Scrum Master", "Google Data Analytics", "Azure Fundamentals"]



# =============================================================================
# SAFE ENTITY ADDER (REGEX + FILTER LABELS)
# =============================================================================

def add_entity(text, value, label, entities):

    # 🔥 FILTRE STRICT LABELS AUTORISÉS
    if label not in TARGET_LABELS:
        return

    if not value:
        return

    pattern = re.escape(value)

    for match in re.finditer(pattern, text, flags=re.IGNORECASE):
        start, end = match.start(), match.end()

        if start >= 0 and end > start:
            entities.append((start, end, label))


def random_date():
    y1 = RNG.randint(2015, 2021)
    y2 = RNG.randint(2022, 2025)
    return f"{RNG.choice(MONTHS)} {y1} - {RNG.choice(MONTHS)} {y2}"


# =============================================================================
# GENERATION
# =============================================================================

synthetic = []
seen = set()

while len(synthetic) < TARGET:

    name = f"{RNG.choice(FIRST_NAMES)} {RNG.choice(LAST_NAMES)}"
    title = RNG.choice(JOB_TITLES)
    company = RNG.choice(COMPANIES)
    degree = RNG.choice(DEGREES)
    institution = RNG.choice(INSTITUTIONS)

    skills = RNG.sample(SKILLS, RNG.randint(4, 8))
    exp = RNG.choice(EXPERIENCE_DESC)
    cert = RNG.choice(CERTIFICATIONS)
    date = random_date()

    template = RNG.randint(1, 4)

    if template == 1:
        text = f"""
{name}
professional summary
{title}
experience
{title} at {company}
{date}
{exp}
skills
{", ".join(skills)}
education
{degree}
{institution}
certification
{cert}
"""

    elif template == 2:
        text = f"""
{name}
skills
{", ".join(skills)}
work history
{title}
{company}
{date}
{exp}
education
{degree}
{institution}
"""

    elif template == 3:
        text = f"""
{name}
education
{degree}
{institution}
experience
{title}
{company}
{date}
{exp}
certification
{cert}
skills
{", ".join(skills)}
"""

    else:
        text = f"""
{name}
{title}
{company}
{date}
{exp}
skills
{", ".join(skills)}
"""

    # normalize
    text = re.sub(r"\n+", "\n", text).strip()

    fp = text_fingerprint(text)

    if fp in seen:
        continue

    seen.add(fp)

    entities = []

    # =============================================================================
    # ENTITY EXTRACTION (STRICT LABEL CONTROL)
    # =============================================================================

    add_entity(text, name, "NAME", entities)  # ❌ sera ignoré automatiquement
    add_entity(text, title, "JOB_TITLE", entities)
    add_entity(text, company, "COMPANY", entities)
    add_entity(text, exp, "EXPERIENCE_DESC", entities)
    add_entity(text, degree, "DEGREE", entities)
    add_entity(text, institution, "INSTITUTION", entities)
    add_entity(text, cert, "CERTIFICATION", entities)
    add_entity(text, date, "DATE", entities)

    for skill in skills:
        add_entity(text, skill, "SKILL", entities)

    # remove duplicates (order-safe)
    seen_local = set()
    clean_entities = []

    for e in entities:
        if e not in seen_local:
            clean_entities.append(e)
            seen_local.add(e)

    synthetic.append({
        "text": text,
        "entities": clean_entities,
        "source": "SYNTHETIC"
    })

print("Synthetic générés :", len(synthetic))

# =============================================================================
# SAUVEGARDE
# =============================================================================

synthetic_path = os.path.join(OUT2, "bloc2_synthetic.json")

with open(synthetic_path, "w", encoding="utf-8") as f:
    json.dump(synthetic, f, ensure_ascii=False, indent=2)

print("💾 Sauvegardé :", synthetic_path)

Synthetic générés : 50
💾 Sauvegardé : /content/drive/MyDrive/cvextraction/Datasets/bloc2_processed/bloc2_synthetic.json


In [ ]:
#@title ⚖️ Bloc3 — fusion F1-MAX (Doccano prioritaire)

from collections import Counter
import copy

# ============================================================
# MODE DATASET — "doccano_only" recommandé pour F1 >= 0.85
# ============================================================
# "doccano_only"  : uniquement S3 Doccano (~400 CV) — MEILLEUR F1
# "doccano_plus"  : Doccano + S1 non dupliqués (sans S2 bruité)
# "full"          : toutes sources (S1+S2+S3+synthetic)

TRAIN_MODE = "doccano_only"

MIN_SPAN_CHARS = 2
MAX_SPAN_CHARS = 500
RARE_LABELS = {"CERTIFICATION", "INSTITUTION", "DEGREE"}
SOURCE_PRIORITY = {"S3_DOCCANO": 3, "S1_DATATURKS": 2, "S2_HEURISTIC": 1, "SYNTHETIC": 0}
LABEL_PRIORITY = {
    "EXPERIENCE_DESC": 8, "JOB_TITLE": 7, "COMPANY": 6,
    "DEGREE": 5, "INSTITUTION": 5, "CERTIFICATION": 5,
    "DATE": 4, "SKILL": 3,
}


def trim_entity_bounds(text, start, end):
    start, end = int(start), int(end)
    if start < 0 or end <= start or end > len(text):
        return None
    raw = text[start:end]
    start += len(raw) - len(raw.lstrip())
    end -= len(raw) - len(raw.rstrip())
    if end <= start or end - start < MIN_SPAN_CHARS or end - start > MAX_SPAN_CHARS:
        return None
    return start, end


def resolve_overlaps(entities):
    if not entities:
        return []
    ordered = sorted(
        entities,
        key=lambda e: (-(LABEL_PRIORITY.get(e[2], 0)), -(e[1] - e[0]), e[0]),
    )
    kept, occupied = [], []
    for start, end, label in ordered:
        if any(not (end <= a or start >= b) for a, b in occupied):
            continue
        kept.append((start, end, label))
        occupied.append((start, end))
    return sorted(kept)


def quality_clean_entities(text, entities):
    cleaned = []
    for start, end, label in entities:
        label = str(label).strip().upper()
        if label not in TARGET_LABELS:
            continue
        bounds = trim_entity_bounds(text, start, end)
        if bounds is None:
            continue
        cleaned.append((bounds[0], bounds[1], label))
    return resolve_overlaps(sorted(set(cleaned)))


# Filtrer entités d'entraînement (6 labels NER — pas CERTIFICATION ni EXPERIENCE_DESC)
NER_TRAIN_LABELS = {
    "SKILL", "JOB_TITLE", "COMPANY", "DATE", "DEGREE", "INSTITUTION",
}
EXCLUDED_LABELS = {"CERTIFICATION", "EXPERIENCE_DESC"}


def filter_train_entities(entities):
    out = []
    for s, e, lbl in entities:
        lbl = str(lbl).strip().upper()
        if lbl not in NER_TRAIN_LABELS:
            continue
        out.append((s, e, lbl))
    return resolve_overlaps(sorted(set(out)))


def clean_cv_record(cv):
    text = clean_text(cv["text"])
    entities = filter_train_entities(quality_clean_entities(text, cv["entities"]))
    if not entities:
        return None
    return {"text": text, "entities": entities, "source": cv.get("source", "UNKNOWN")}


def merge_by_fingerprint(cv_list):
    by_fp = {}
    for cv in cv_list:
        rec = clean_cv_record(cv)
        if rec is None:
            continue
        fp = text_fingerprint(rec["text"])
        prev = by_fp.get(fp)
        if prev is None or SOURCE_PRIORITY.get(rec["source"], 0) > SOURCE_PRIORITY.get(prev["source"], 0):
            by_fp[fp] = rec
    return list(by_fp.values())


# --- Sélection sources selon mode ---
if TRAIN_MODE == "doccano_only":
    raw_pool = list(processed_s3)
elif TRAIN_MODE == "doccano_plus":
    s1_no_dup = merge_by_fingerprint(processed_s1)
    s3_fps = {text_fingerprint(clean_text(c["text"])) for c in processed_s3}
    s1_extra = [c for c in s1_no_dup if text_fingerprint(c["text"]) not in s3_fps]
    raw_pool = list(processed_s3) + s1_extra
else:
    raw_pool = processed_s1 + processed_s2 + processed_s3 + synthetic

bloc3 = merge_by_fingerprint(raw_pool)

label_counts = Counter()
source_counts = Counter()
for cv in bloc3:
    source_counts[cv["source"]] += 1
    for _, _, lbl in cv["entities"]:
        label_counts[lbl] += 1

print("=" * 50)
print(f"MODE : {TRAIN_MODE}")
print(f"CV   : {len(bloc3)}")
print("=" * 50)
print("Par source :")
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"  {src:20s} {cnt}")
print("\nPar label :")
for lbl, cnt in sorted(label_counts.items()):
    print(f"  {lbl:20s} {cnt}")

natural_path = os.path.join(OUT3, "bloc3_natural.json")
with open(natural_path, "w", encoding="utf-8") as f:
    json.dump(bloc3, f, ensure_ascii=False)
print("\nSauvegardé :", natural_path)


MODE : doccano_only
CV   : 398
Par source :
  S3_DOCCANO           398

Par label :
  CERTIFICATION        197
  COMPANY              992
  DATE                 1421
  DEGREE               383
  EXPERIENCE_DESC      962
  INSTITUTION          377
  JOB_TITLE            1183
  SKILL                5190

Sauvegardé : /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_natural.json


In [8]:
#@title 📦 Split train/dev — 6 labels NER (RoBERTa-large)

from sklearn.model_selection import train_test_split
from collections import Counter
import copy

DEV_RATIO = 0.15
LABEL_BOOST = {
    "INSTITUTION": 4,
    "COMPANY": 3,
    "SKILL": 2,
    "DEGREE": 2,
}
DEFAULT_BOOST = 1


def doc_boost_factor(cv):
    labels = {lbl for _, _, lbl in cv["entities"]}
    return max([DEFAULT_BOOST] + [LABEL_BOOST.get(l, DEFAULT_BOOST) for l in labels])


def build_strat_label(cv):
    labels = sorted(set(lbl for _, _, lbl in cv["entities"]))
    return labels[0] if labels else "EMPTY"


strat_labels = [build_strat_label(cv) for cv in bloc3]
freq = Counter(strat_labels)
safe_labels = [lbl if freq[lbl] >= 2 else "OTHER" for lbl in strat_labels]

FIXED_TRAIN_CACHE = os.path.join(OUT3, "doccano_fixed_split_train.json")
FIXED_DEV_CACHE = os.path.join(OUT3, "doccano_fixed_split_dev.json")

_fixed_train = _fixed_dev = None
if globals().get("USE_FIXED_DOCCANNO_SPLIT") and globals().get("PRE_SPLIT_TRAIN") is not None:
    _fixed_train, _fixed_dev = PRE_SPLIT_TRAIN, PRE_SPLIT_DEV
elif os.path.isfile(FIXED_TRAIN_CACHE) and os.path.isfile(FIXED_DEV_CACHE):
    with open(FIXED_TRAIN_CACHE, encoding="utf-8") as f:
        _fixed_train = json.load(f)
    with open(FIXED_DEV_CACHE, encoding="utf-8") as f:
        _fixed_dev = json.load(f)

_doccano_train_on_disk = any(
    os.path.isfile(p)
    for p in (
        os.path.join(OUT, "TRAIN.jsonl"),
        os.path.join(OUT, "train.jsonl"),
        os.path.join(OUT3, "TRAIN.jsonl"),
    )
)

if _fixed_train is not None and _fixed_dev is not None:
    train_cv = [clean_cv_record(cv) for cv in _fixed_train if clean_cv_record(cv)]
    dev_cv = [clean_cv_record(cv) for cv in _fixed_dev if clean_cv_record(cv)]
    src = "TRAIN.jsonl + DEV.jsonl" if globals().get("USE_FIXED_DOCCANNO_SPLIT") else "cache Drive"
    print(f"Split : FIXE Doccano ({src})")
elif _doccano_train_on_disk:
    raise RuntimeError(
        "TRAIN.jsonl / DEV.jsonl détectés sur Drive mais split fixe non chargé.\n"
        "→ Ré-exécute la cellule « S3 — Import Doccano » puis cette cellule."
    )
else:
    train_cv, dev_cv = train_test_split(
        bloc3, test_size=DEV_RATIO, random_state=SEED, stratify=safe_labels,
    )
    print("Split : aléatoire stratifié (15% dev)")
    print("⚠️ Pas de TRAIN.jsonl/DEV.jsonl — corrections Doccano NON appliquées")


def oversample(dataset):
    out = list(dataset)
    for cv in dataset:
        copies = doc_boost_factor(cv) - 1
        for _ in range(copies):
            out.append(copy.deepcopy(cv))
    return out


train_balanced = oversample(train_cv)
balanced_bloc3 = train_balanced + dev_cv

print("Labels entraînement :", sorted(NER_TRAIN_LABELS))
print("Exclus (LLM / retiré) :", sorted(EXCLUDED_LABELS))
print(f"Train {len(train_cv)} → balanced {len(train_balanced)} | Dev {len(dev_cv)}")

for name, data in [("TRAIN", train_balanced), ("DEV", dev_cv)]:
    c = Counter()
    for cv in data:
        for _, _, lbl in cv["entities"]:
            c[lbl] += 1
    print(f"\n{name}:")
    for lbl, n in sorted(c.items()):
        print(f"  {lbl:20s} {n}")

# --- Manifestes pour audit Doccano (60 CV dev) ---
import json


def cv_manifest_row(idx, cv, split):
    fp = text_fingerprint(cv["text"])
    labels = Counter(lbl for _, _, lbl in cv["entities"])
    preview = cv["text"][:160].replace("\n", " ").strip()
    return {
        "idx": idx,
        "split": split,
        "fingerprint": fp,
        "search_hint": preview,
        "n_entities": int(sum(labels.values())),
        "labels": dict(labels),
    }


dev_manifest = [cv_manifest_row(i, cv, "dev") for i, cv in enumerate(dev_cv)]
train_manifest = [cv_manifest_row(i, cv, "train") for i, cv in enumerate(train_cv)]


def export_cv_record(cv):
    """6 labels NER uniquement — prêt pour audit Doccano."""
    text = clean_text(cv["text"])
    entities = filter_train_entities(quality_clean_entities(text, cv["entities"]))
    return {
        "text": text,
        "entities": entities,
        "source": cv.get("source", "UNKNOWN"),
        "fingerprint": text_fingerprint(text),
    }


dev_export = [export_cv_record(cv) for cv in dev_cv]
train_export = [export_cv_record(cv) for cv in train_cv]

for fname, payload in [
    ("dev_manifest.json", dev_manifest),
    ("train_manifest.json", train_manifest),
    ("dev_cv.json", dev_export),
    ("train_cv.json", train_export),
]:
    path = os.path.join(OUT3, fname)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2 if fname.endswith("manifest.json") else None)
    print("Sauvegardé :", path, f"({len(payload)} CV)" if "cv.json" in fname else "")


Labels entraînement : ['COMPANY', 'DATE', 'DEGREE', 'INSTITUTION', 'JOB_TITLE', 'SKILL']
Exclus (LLM / retiré) : ['CERTIFICATION', 'EXPERIENCE_DESC']
Train 338 → balanced 1291 | Dev 60

TRAIN:
  CERTIFICATION        666
  COMPANY              3262
  DATE                 4704
  DEGREE               1284
  EXPERIENCE_DESC      3218
  INSTITUTION          1288
  JOB_TITLE            3821
  SKILL                16559

DEV:
  CERTIFICATION        28
  COMPANY              147
  DATE                 215
  DEGREE               57
  EXPERIENCE_DESC      131
  INSTITUTION          55
  JOB_TITLE            189
  SKILL                848


In [9]:
#@title 🤖 Conversion spaCy (DocBin) — dataset final

import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans
import os

nlp = spacy.blank("en")


def to_docbin(cv_list, name=""):
    db = DocBin(attrs=["ENT_IOB", "ENT_TYPE"], store_user_data=False)
    discarded = 0
    docs = 0

    for cv in cv_list:
        doc = nlp.make_doc(cv["text"])
        spans = []

        for start, end, label in cv["entities"]:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                span = doc.char_span(start, end, label=label, alignment_mode="expand")
            if span is None:
                discarded += 1
                continue
            spans.append(span)

        spans = filter_spans(spans)
        if not spans:
            continue
        doc.ents = spans
        db.add(doc)
        docs += 1

    print(f"  {name}: {docs} docs | spans ignorés: {discarded}")
    return db


print("Conversion DocBin…")
train_db = to_docbin(train_balanced, "train")
dev_db = to_docbin(dev_cv, "dev")

TRAIN_PATH = os.path.join(OUT3, "bloc3_balanced_train.spacy")
DEV_PATH = os.path.join(OUT3, "bloc3_balanced_dev.spacy")

train_db.to_disk(TRAIN_PATH)
dev_db.to_disk(DEV_PATH)

with open(os.path.join(OUT3, "bloc3_balanced.json"), "w", encoding="utf-8") as f:
    json.dump(balanced_bloc3, f, ensure_ascii=False)

print("\n====================================")
print("EXPORT FINAL")
print("====================================")
print("Train :", TRAIN_PATH)
print("Dev   :", DEV_PATH)
print("\n✅ Dataset prêt pour notebook 06")


Conversion DocBin…
  train: 1291 docs | spans ignorés: 0
  dev: 60 docs | spans ignorés: 0

EXPORT FINAL
Train : /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_balanced_train.spacy
Dev   : /content/drive/MyDrive/cvextraction/Datasets/bloc3_processed/bloc3_balanced_dev.spacy

✅ Dataset prêt pour notebook 06
